To run this, press "*Run*" and press "*Run All*" on your local **Intel GPU**!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **[Unsloth Desktop](https://unsloth.ai/docs/desktop)**, the first desktop app to run and train models. Free and open-source for macOS, Windows and Linux. [GitHub](https://github.com/unslothai/unsloth) • [Download](https://unsloth.ai/download)

<p>
<a href="https://unsloth.ai/docs/desktop"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/unsloth-qwen3-8.png" width="350" alt="Introducing Unsloth Desktop"></a>
</p>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
# vLLM's XPU wheel requires one exact torch, so it goes first and brings that torch.
!pip install --upgrade -qqq uv
!uv pip install -qqq "vllm==0.30.0" "torch==2.13.0" "torchvision==0.28.0" "transformers>=5.10.4,<5.16" --extra-index-url https://wheels.vllm.ai/0.30.0/xpu --extra-index-url https://wheels.vllm.ai/xpu --extra-index-url https://download.pytorch.org/whl/xpu --index-strategy=unsafe-best-match
# torch and triton are already satisfied, so uv keeps the XPU builds.
!uv pip install -qqq "bitsandbytes>=0.50.0" unsloth unsloth_zoo
try:
    import importlib.metadata as _md; _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
except Exception:
    _torch_v = ()
# torchao 0.18.0 imports torch.nn.functional.ScalingType, added in torch 2.10; peft >= 0.19 needs the 0.16.0 floor.
_torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
!uv pip install -qqq --no-deps --index-url https://download.pytorch.org/whl/xpu "{_torchao}"
!uv pip install -qqq --no-deps -U "tokenizers>=0.22.0,<=0.23.0"


In [2]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!


### Unsloth

Goal: To convert `Qwen3-4B-Base` into a reasoning model via GRPO by using OpenR1's Math dataset.

We first pre fine-tune the model to make GRPO skip trying to match formatting - this speeds GRPO up.

In [3]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Base",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/path/to/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1019 05:42:21.450000 19655 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1019 05:42:21.480000 19655 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


2026-10-19 05:42:27,589	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


WARNING 10-19 05:42:32 [indexer_topk.py:29] Failed to import the DeepSelect extension (vllm._deepselect_C): No module named 'vllm._deepselect_C'
INFO 10-19 05:42:32 [vllm_utils.py:888] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.10.6: Fast Qwen3 patching. Transformers: 5.15.1. vLLM: 0.30.0+xpu.
   \\   /|    Intel(R) Arc(TM) Pro B60 Graphics. Num GPUs = 4. Max memory: 23.906 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+xpu. Intel Toolkit: 20260000. Triton: 3.7.2
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

`torch_dtype` is deprecated! Use `dtype` instead!


Unsloth: Just some info: will skip parsing ['layer_norm2', 'post_feedforward_layernorm_1', 'ffn_norm', 'post_feedforward_layernorm_2', 'pre_feedforward_layernorm_2', 'post_layernorm', 'input_layernorm', 'pre_feedforward_layernorm', 'post_feedforward_layernorm', 'layer_norm1', 'q_norm', 'post_per_layer_input_norm', 'norm2', 'operator_norm', 'norm', 'k_norm', 'q_layernorm', 'attention_norm', 'norm1', 'post_attention_layernorm', 'k_layernorm']


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 12599.31it/s]


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['layer_norm2', 'post_feedforward_layernorm_1', 'ffn_norm', 'post_feedforward_layernorm_2', 'pre_feedforward_layernorm_2', 'post_layernorm', 'input_layernorm', 'pre_feedforward_layernorm', 'post_feedforward_layernorm', 'layer_norm1', 'q_norm', 'post_per_layer_input_norm', 'norm2', 'operator_norm', 'norm', 'k_norm', 'q_layernorm', 'cross_attn_post_attention_layernorm', 'attention_norm', 'norm1', 'cross_attn_input_layernorm', 'post_attention_layernorm', 'k_layernorm']


Unsloth 2026.10.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### GRPO chat template
Since we're using a base model, we should set a chat template. You can make your own chat template as well!
1. DeepSeek uses `<think>` and `</think>`, but this is **not** necessary - you can customize it however you like!
2. A `system_prompt` is recommended to at least guide the model's responses.

In [4]:
reasoning_start = "<start_working_out>" # Acts as think-open tag
reasoning_end   = "<end_working_out>"   # Acts as think-close tag
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>'

We create a simple chat template below. Notice `add_generation_prompt` includes prepending `<start_working_out>` to guide the model to start its reasoning process.

In [5]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

# Replace with our specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

Let's see how our chat template behaves on an example:

In [6]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION><|endoftext|>What is 1+1?<start_working_out>I think it's 2.<end_working_out><SOLUTION>2</SOLUTION><|endoftext|>What is 2+2?<start_working_out>"

### Pre fine-tuning for formatting
We now use a subset of NVIDIA's [Open Math Reasoning dataset](https://huggingface.co/datasets/nvidia/OpenMathReasoning) which was filtered to only include high quality DeepSeek R1 traces.

We'll only filter ~59 or so examples to first "prime" / pre fine-tune the model to understand our custom GRPO formatting.

In [7]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
dataset = dataset.to_pandas()[
    ["expected_answer", "problem", "generated_solution"]
]

# Try converting to number - if not, replace with NaN
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors = "coerce").notnull()
# Select only numbers
dataset = dataset.iloc[np.where(is_number)[0]]

dataset

,expected_answer,problem,generated_solution
0,14,Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$...,"<think>\nOkay, let's see. I need to solve the ..."
6,-2,Find the value of the parameter $a$ for which ...,"<think>\nOkay, so I need to find the value of ..."
9,18,What is the sum of all real numbers $x$ for wh...,"<think>\nOkay, so I need to solve the equation..."
13,2,Evaluate the sum \(\sum_{n=1}^\infty \frac{\ph...,"<think>\nOkay, so I need to evaluate the infin..."
17,30,What is the largest positive integer that divi...,"<think>\nAlright, so I need to find the larges..."
...,...,...,...
19243,244,"Let \( p \), \( q \), and \( r \) be the disti...","<think>\nOkay, so I need to find the value of ..."
19245,1,A bug is on the $0$ of a number line. At any p...,"<think>\nOkay, so I have this problem where a ..."
19247,4,A bus left point X for point Y. Two hours late...,"<think>\nOkay, let's tackle this problem step ..."
19248,18,Each interior angle of a regular n-gon measure...,"<think>\nOkay, let's see. I need to find the n..."


We have to format the dataset to follow our GRPO style formatting:

In [8]:
def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]

    # Remove generated think tags
    thoughts = x["generated_solution"]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")

    # Strip newlines on left and right
    thoughts = thoughts.strip()
    # Add our custom formatting
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis = 1)

Check to see if it worked:

In [9]:
tokenizer.apply_chat_template(dataset["Messages"][0], tokenize = False)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION><|endoftext|>Given $\\sqrt{x^2+165}-\\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<start_working_out>Okay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.\n\nFirst, let me write down the equation again to make sure I have it right:\n\n√(x² + 165) - √(x² - 52) = 7.\n\nOkay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:\n\n√(x² + 165) = 7 + √(x² - 52).\n\nNow, if I square both sides, maybe I can get rid of the square roots. Let's do that:\n\n(√(x² + 165))² = (7 + √(x² - 52))².\n\nSimplifying the left side:\n\nx² + 165 = 49 + 14√(x² - 52) + (√(x² - 52))².\n\nThe right side is expanded using the formula (a + b)² = a² + 2ab + b². So the right side becomes 7² + 2*7*√(x² - 52) + (√(x² - 52))², which is 49 + 14√(x² - 52) + (x² - 52).\n\nSo putting it all together:\n\nx² + 165 = 49 + 14√(x² - 52) + x² - 52.\n\nHmm, let's simplify the right side. The x² terms will cancel out, right? Let's subtract x² from both sides:\n\n165 = 49 + 14√(x² - 52) - 52.\n\nSimplify the constants on the right:\n\n49 - 52 is -3, so:\n\n165 = -3 + 14√(x² - 52).\n\nNow, add 3 to both sides to isolate the radical term:\n\n165 + 3 = 14√(x² - 52).\n\nSo 168 = 14√(x² - 52).\n\nDivide both sides by 14:\n\n168 / 14 = √(x² - 52).\n\n12 = √(x² - 52).\n\nNow, square both sides again to eliminate the square root:\n\n12² = x² - 52.\n\n144 = x² - 52.\n\nAdd 52 to both sides:\n\n144 + 52 = x².\n\n196 = x².\n\nSo x = √196 = 14.\n\nBut wait, since the problem states that x is positive, we only take the positive root. So x = 14.\n\nBut hold on, when dealing with squaring equations, sometimes extraneous solutions can come up. I should check if this solution actually satisfies the original equation.\n\nLet's plug x = 14 back into the original equation:\n\n√(14² + 165) - √(14² - 52) = ?\n\nCalculate each term:\n\n14² is 196.\n\nSo first radical: √(196 + 165) = √361 = 19.\n\nSecond radical: √(196 - 52) = √144 = 12.\n\nSo 19 - 12 = 7, which is exactly the right-hand side. So yes, it checks out.\n\nTherefore, the only solution is x = 14. Since the problem says x is positive, we don't have to consider negative roots. So I think that's the answer.\nTo solve the equation \\(\\sqrt{x^2 + 165} - \\sqrt{x^2 - 52} = 7\\) for positive \\(x\\), we proceed as follows:\n\n1. Start with the given equation:\n   \\[\n   \\sqrt{x^2 + 165} - \\sqrt{x^2 - 52} = 7\n   \\]\n\n2. Isolate one of the square roots by moving \\(\\sqrt{x^2 - 52}\\) to the right side:\n   \\[\n   \\sqrt{x^2 + 165} = 7 + \\sqrt{x^2 - 52}\n   \\]\n\n3. Square both sides to eliminate the square root on the left:\n   \\[\n   (\\sqrt{x^2 + 165})^2 = (7 + \\sqrt{x^2 - 52})^2\n   \\]\n   Simplifying both sides, we get:\n   \\[\n   x^2 + 165 = 49 + 14\\sqrt{x^2 - 52} + (x^2 - 52)\n   \\]\n\n4. Combine like terms on the right side:\n   \\[\n   x^2 + 165 = x^2 - 52 + 49 + 14\\sqrt{x^2 - 52}\n   \\]\n   Simplifying further:\n   \\[\n   x^2 + 165 = x^2 - 3 + 14\\sqrt{x^2 - 52}\n   \\]\n\n5. Subtract \\(x^2\\) from both sides:\n   \\[\n   165 = -3 + 14\\sqrt{x^2 - 52}\n   \\]\n\n6. Add 3 to both sides to isolate the term with the square root:\n   \\[\n   168 = 14\\sqrt{x^2 - 52}\n   \\]\n\n7. Divide both sides by 14:\n   \\[\n   12 = \\sqrt{x^2 - 52}\n   \\]\n\n8. Square both sides again to eliminate the square root:\n   \\[\n   12^2 = x^2 - 52\n   \\]\n   Simplifying:\n   \\[\n   144 = x^2 - 52\n   \\]\n\n9. Add 52 to both sides to solve for \\(x^2\\):\n   \\[\n   196 = x^2\n   \\]\n\n10. Take the positive square root (since \\(x\\) is positive):\n    \\[\n    x =

Let's truncate the pre fine-tuning dataset to `max_seq_length/2` since we don't want too long reasoning traces.

Note this might take 2 minutes!

In [10]:
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))

dataset = dataset.loc[dataset["N"] <= max_seq_length/2].copy()
dataset.shape

(59, 5)

We then tokenize the messages and convert it to a Hugging Face compatible dataset format:

In [11]:
from datasets import Dataset

dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)
dataset

Dataset({
    features: ['expected_answer', 'problem', 'generated_solution', 'Messages', 'N', 'text', '__index_level_0__'],
    num_rows: 59
})

Let's now pre fine-tune the model so it follows our custom GRPO formatting!

In [12]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8): 100%|██████████| 59/59 [00:13<00:00,  4.42 examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [13]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 59 | Num Epochs = 2 | Total steps = 118
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


2026:10:16-00:09:25:3714823 |CCL_WARN| value of CCL_ATL_TRANSPORT changed to be ofi (default:mpi)
2026:10:16-00:09:25:3714823 |CCL_WARN| could not get local_idx/count from environment variables, trying to get them from ATL
Unsloth: Will smartly offload gradients to save VRAM!
WARNING 10-16 00:09:26 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _rms_layernorm_forward. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 10-16 00:09:27 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _rms_layernorm_backward. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 10-16 00:09:27 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _rms_layernorm_forward_rows. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 10-16 00:09:27 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _rms_layernorm_backward_rows

/path/to/venv/lib/python3.12/site-packages/unsloth/kernels/rms_layernorm.py:521: UserWarning: Unsloth: narrow-row RMSNorm kernels disabled (self-check mismatch for (device(type='xpu', index=0), torch.bfloat16, torch.bfloat16, 128, 1e-06, False)); using one row per program.
  warnings.warn(


WARNING 10-16 00:09:28 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _fg_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 10-16 00:09:31 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _DWf_DW_dfg_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 10-16 00:09:33 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _optimizer_update_2state_8bit_blockwise_triton_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Step,Training Loss
5,0.654437
10,0.656774
15,0.425033
20,0.390873
25,0.423448
30,0.452799
35,0.475927
40,0.418529
45,0.444869
50,0.330500


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


TrainOutput(global_step=118, training_loss=0.3501755453772464, metrics={'train_runtime': 85.3718, 'train_samples_per_second': 1.382, 'train_steps_per_second': 1.382, 'total_flos': 2374193056317440.0, 'train_loss': 0.3501755453772464, 'epoch': 2.0})

Let's check if the model has learnt to follow the custom format:

In [14]:
text = tokenizer.apply_chat_template(
    dataset[0]["Messages"][:2],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("xpu"),
    temperature = 0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

You are given a problem.
Think about the problem and provide your working out.
Place it between <start_working_out> and <end_working_out>.
Then, provide your solution between <SOLUTION></SOLUTION><|endoftext|>Jenifer has 82 cents in pennies and nickels. Her younger brother mistook all her nickels for dimes and counted the total as $1.47. How many pennies does Jenifer

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


have?<start_working_out>Okay, let's see. Jenifer has 82 cents in pennies and nickels. Her brother thought all the nickels were dimes and counted the total as $1.47. I need to find out how many pennies she has. Hmm, let's break this down.

First, I need to set up some equations. Let's say the number of pennies is P and the number of nickels is N. Since pennies are worth 1 cent and nickels are worth 5 cents, the total value is 1P + 5N = 82 cents. That's the first equation.

Now, her brother thought all the nickels were dimes. Dimes are worth 10 cents. So, he counted the total as $1.47, which is 147 cents. So, the equation for his count would be P + 10N = 147. That's the second equation.

So now I have two equations:
1. P + 5N = 82
2. P + 10N = 147

I need to solve these two equations to find P. Let me think. Maybe I can subtract the first equation from the second to eliminate P. Let's try that.

Subtracting equation 1 from equation 2:
(P + 10N) - (P + 5N) = 147 - 82
That simplifies to 5N = 65
So, N = 65 / 5 = 13

So there are 13 nickels. Now, plug N back into the first equation to find P:
P + 5*13 = 82
P + 65 = 82
P = 82 - 65 = 17

So Jenifer has 17 pennies. Let me check that. If she has 17 pennies and 13 nickels, that's 17 + 65 = 82 cents, which matches. Her brother thought the nickels were dimes, so that's 17 + 130 = 147 cents, which is $1.47. That checks out. So the answer should be 17 pennies.
To solve the problem, we start by setting up two equations based on the given information:

1. The total value of the coins in cents is 82:
   \[
   P + 5N = 82
   \]
   where \(P\) is the number of pennies and \(N\) is the number of nickels.

2. The total value counted by Jenifer's brother, assuming all nickels are dimes, is 147 cents:
   \[
   P + 10N = 147
   \]

We now have the system of linear equations:
\[
\begin{cases}
P + 5N = 82 \\
P + 10N = 147
\end{cases}
\]

To eliminate \(P\), we subtract the first equation from the second:
\[
(P + 10N) - (P + 5N) = 147 - 82
\]
\[
5N = 65
\]
\[
N = 13
\]

Now that we have \(N = 13\), we substitute this value back into the first equation to find \(P\):
\[
P + 5(13) = 82
\]
\[
P + 65 = 82
\]
\[
P = 17
\]

Thus, Jenifer has \(\boxed{17}\) pennies.<end_working_out><SOLUTION>17</SOLUTION><|endoftext|>

Yes it did follow the formatting! Great! Let's remove some items before the GRPO step

In [15]:
del dataset
torch.xpu.empty_cache()
import gc
gc.collect()

0

### Data Prep
<a name="Data"></a>

We're using Hugging Face's [Open R1 Math dataset](https://huggingface.co/datasets/open-r1/DAPO-Math-17k-Processed). You can also utilize OpenAI's famous [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)

In [16]:
from datasets import load_dataset
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split = "train")
dataset

Generating train split: 100%|██████████| 14116/14116 [00:00<00:00, 206938.62 examples/s]


Dataset({
    features: ['prompt', 'solution', 'data_source', 'source_prompt', 'ability', 'reward_model', 'extra_info'],
    num_rows: 14116
})

Let's look at the first row:

In [17]:
dataset[0]["prompt"]

'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.'

In [18]:
dataset[0]["solution"]

'34'

In GSM8K, we notice all answers like about have a ####, so we extract it. But for the Open R1 dataset, we can skip the below.

In [19]:
def extract_hash_answer(text):
    # if "####" not in text: return None
    # return text.split("####")[1].strip()
    return text
extract_hash_answer(dataset[0]["solution"])

'34'

Let's map the dataset! and see the first row:

In [20]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})
dataset[0]

Map: 100%|██████████| 14116/14116 [00:00<00:00, 18126.43 examples/s]


{'prompt': [{'role': 'system',
   'content': 'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>'},
  {'role': 'user',
   'content': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.'}],
 'solution': '34',
 'data_source': 'math_dapo',
 'source_prompt': [{'content': 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nIn triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.\n\nRemember to put your answer on its own line after "Answer:".',
   'role': 'user'}],
 'ability': 'MATH',
 'reward_model': {'ground_truth': '34', 'style': 'rule-lighteval/MATH_v2'},
 'extra_info': {'index': '9a9b6eb4-a1cb-49d1-8c1e-62eaf2f74079'},
 'answer': '34'}

We create a regex format to match the reasoning sections and answers:

In [21]:
import re

# Add optional EOS token matching
solution_end_regex = r"</SOLUTION>[\s]{0,}" + \
    "(?:" + re.escape(tokenizer.eos_token) + ")?"

match_format = re.compile(
    rf"{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end_regex}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)
print(match_format)

re.compile('<end_working_out>.*?<SOLUTION>(.+?)</SOLUTION>[\\s]{0,}(?:<\\|endoftext\\|>)?[\\s]{0,}$', re.MULTILINE|re.DOTALL)

We verify it works:

In [22]:
match_format.findall(
    "Let me think!<end_working_out>"\
    f"<SOLUTION>\n2\n</SOLUTION>",
)

['\n2\n']

In [23]:
match_format.findall(
    "<start_working_out>Let me think!<end_working_out>"\
    f"<SOLUTION>  2  </SOLUTION>\n\n",
)

['  2  ']

We now want to create a reward function to match the format exactly - we reward it with 3 points if it succeeds:

In [24]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

If it fails, we want to reward the model if it at least follows the format partially, by counting each symbol:

In [25]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!

        # No need to reward the opening tag since we always prepend it!
        # score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        score += 0.5 if response.count(solution_start)  == 1 else -1.0
        score += 0.5 if response.count(solution_end)    == 1 else -1.0
        scores.append(score)
    return scores

Finally, we want to extract the generated answer, and reward or penalize it! We also reward it based on how close the answer is to the true one via ratios:

In [26]:
def check_answer(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(-2.0)
            continue
        # Correct answer gets 5 points!
        if guess == true_answer:
            score += 5.0
        # Match if spaces are seen, but less reward
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # We also reward it if the answer is close via ratios!
            # Ie if the answer is within some range, reward it!
            try:
                ratio = float(guess) / float(true_answer)
                if   ratio >= 0.9 and ratio <= 1.1: score += 2.0
                elif ratio >= 0.8 and ratio <= 1.2: score += 1.5
                else: score -= 2.5 # Penalize wrong answers
            except:
                score -= 4.5 # Penalize
        scores.append(score)
    return scores

Also sometimes it might not be 1 number as the answer, but like a sentence for example "The solution is $20" -> we extract 20.

We also remove possible commas for example as in 123,456

In [27]:
match_numbers = re.compile(
    solution_start + r".*?[\s]{0,}([-]?[\d\.\,]{1,})",
    flags = re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  -0.234  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>17</SOLUTION>"))

['0.34']
['123,456']
['-0.234']
['17']


We now prepare our main function which will print out the generated responses and the true answer, along with another reward function which converts text to float via `float` and sees if it's the same.

In [28]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_numbers.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    # Print only every few steps
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}"
        )
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        # Convert to numbers
        try:
            true_answer = float(true_answer.strip())
            # Remove commas like in 123,456
            guess       = float(guess.strip().replace(",", ""))
            scores.append(3.5 if guess == true_answer else -1.5)
        except:
            scores.append(0)
            continue
    return scores

Get the top 90% prompt length so we don't accidentally truncate them!

Ie we'll remove the top 10% long prompts.

In [29]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

Map: 100%|██████████| 14116/14116 [00:03<00:00, 4031.55 examples/s]


You are given a problem.
Think about the problem and provide your working out.
Place it between <start_working_out> and <end_working_out>.
Then, provide your solution between <SOLUTION></SOLUTION><|endoftext|>In triangle $ABC$, $\sin \angle A = \frac{4}{5}$ and $\angle A < 90^\circ$. Let $D$ be a point outside triangle $ABC$ such that $\angle BAD = \angle DAC$ and $\angle BDC = 90^\circ$. Suppose that $AD = 1$ and that $\frac{BD}{CD} = \frac{3}{2}$. If $AB + AC$ can be expressed in the form $\frac{a\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.<start_working_out>

Map: 100%|██████████| 14116/14116 [00:01<00:00, 12232.27 examples/s]


Max Length =  201


<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [30]:
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 100,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

Unsloth: GRPO now defaults to TRL's loss_type = 'dapo' with beta = 0.0. Set loss_type = 'bnpo', beta = 0.001 for Unsloth's previous default.
Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4
Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.
Unsloth: `max_prompt_length` is not supported by the installed TRL's GRPOConfig and will be IGNORED - filter overlong prompts out of your dataset before training instead.


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [31]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args = training_args,
    train_dataset = dataset,

    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)
trainer.train()

INFO 10-16 00:12:36 [scheduler.py:2600] setting pause state to PAUSED_NEW
INFO 10-16 00:12:36 [sleep_mode_backend.py:183] Using sleep-mode backend: cumem


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,709 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


INFO 10-16 00:12:37 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:12:37 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:12:37 [scheduler.py:2600] setting pause state to UNPAUSED
WARNING 10-16 00:12:37 [input_processor.py:196] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
WARNING 10-16 00:12:38 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _topk_log_softmax_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
INFO 10-16 00:14:14 [scheduler.py:2600] setting pause state to PAUSED_NEW
Unsloth: Will smartly offload gradients to save VRAM!
WARNING 10-16 00:14:29 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _rope_embedding. This causes a latency spike; consider extendin

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_exactly / mean,rewards / match_format_exactly / std,rewards / match_format_approximately / mean,rewards / match_format_approximately / std,rewards / check_answer / mean,rewards / check_answer / std,rewards / check_numbers / mean,rewards / check_numbers / std
1,0.000000,-7.500000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
2,0.170955,-5.500000,4.000000,1657.750000,1093.000000,1846.000000,0.750000,1093.000000,1093.000000,1093.000000,0.000000,0.750000,1.500000,-1.875000,2.250000,-2.125000,0.250000,-2.250000,0.500000
3,0.000000,-7.500000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
4,0.000000,-7.500000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
5,0.000000,13.000000,0.000000,1258.750000,1075.000000,1363.000000,0.000000,1258.750000,1075.000000,1363.000000,0.000000,3.000000,0.000000,1.500000,0.000000,5.000000,0.000000,3.500000,0.000000
6,0.000000,-7.500000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
7,0.048380,-3.500000,4.618802,1748.500000,1552.000000,1846.000000,0.500000,1651.000000,1552.000000,1750.000000,0.000000,1.500000,1.732051,-0.750000,2.598076,-2.250000,0.288675,-2.000000,0.577350
8,0.127467,2.750000,11.835681,1610.000000,1147.000000,1846.000000,0.500000,1374.000000,1147.000000,1601.000000,0.000000,1.500000,1.732051,-0.750000,2.598076,1.500000,4.041452,0.500000,3.464102
9,0.000000,-7.500000,0.000000,1846.000000,1846.000000,1846.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
10,0.145507,-2.375000,10.250000,1683.250000,1195.000000,1846.000000,0.750000,1195.000000,1195.000000,1195.000000,0.000000,0.750000,1.500000,-1.875000,2.250000,-0.250000,3.500000,-1.000000,3.000000


INFO 10-16 00:15:55 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:15:55 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:15:55 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:17:44 [scheduler.py:2600] setting pause state to PAUSED_NEW
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
INFO 10-16 00:18:01 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:18:01 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:18:01 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:19:55 [scheduler.py:2600] setting pause state to PAUSED_NEW
INFO 10-16 00:20:02 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:20:02 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:20:02 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:21:48 [scheduler.py:2600] setting pause state to PAUSED_NEW
INFO 10-16 00:21:55 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:21:55 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:21:55 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:23:08 [scheduler.py:2600] setting pause state to PAUSED_NEW
INFO 10-16 00:23:14 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:23:14 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:23:14 [scheduler.py:2600] setting pause state to UNPAUSED
INFO 10-16 00:24:51 [scheduler.py:2600] setting pause state to PAUSED_NEW
********************Question:
The sum of $\lfloor x \rfloor$ for all real numbers $x$ satisfying the equation $16 + 15x + 15x^2 = \lfloor x \rfloor^3$ is: 
Answer:
33 
Response:
Okay, let's see. I need to find all real numbers x that satisfy the equation 16 + 15x + 15x² = ⌊x⌋³, where ⌊x⌋ is the floor of x, meaning the greatest integer less than or equal to x. Then, I have to sum all those floor values of x. Hmm, okay. Let me start by trying to understand the equation. The left side is a quadratic in x, and the right side is the cube of the floor of x. Maybe I can consider different integer values for ⌊x⌋ and see if they work.

Let me denote ⌊x⌋ as n, where n is an integer. Then the equation becomes 16 + 15x + 15x² = n³. But since x is between n and n+1 (because of the floor function), maybe I can express x in terms of n and then substitute back into the equation. Let's try that. Let x = n + f, where 0 ≤ f < 1. Then substitute into the equation:

16 + 15(n + f) + 15(n + f)² = n³

Expanding that:

16 + 15n + 15f + 15(n² + 2nf + f²) = n³
16 + 15n + 15f + 15n² + 30nf + 15f² = n³

Now, let me rearrange terms:

15n² + 30nf + 15f² + 15n + 15f + 16 - n³ = 0

Hmm, this seems complicated. Maybe there's a better way. Let me think differently. Since n is an integer, maybe I can test integer values of n and see if the equation holds for some x in that interval. Let's try small integers first.

Let's start with n=0. Then the equation is 16 + 15x + 15x² = 0. Let's solve the quadratic equation:

15x² + 15x + 16 = 0

Using the quadratic formula:

x = [-15 ± sqrt(15² - 4*15*16)] / (2*15)
= [-15 ± sqrt(225 - 960)] / 30
= [-15 ± sqrt(-735)] / 30

Since the discriminant is negative, there are no real solutions here. So n=0 doesn't work.

Next, n=1. The equation becomes:

16 + 15x + 15x² = 1

Rearranged:

15x² + 15x + 15 = 0
Divide by 15: x² + x + 1 = 0

Solutions: x = [-1 ± sqrt(1 - 4)] / 2 = [-1 ± sqrt(-3)] / 2

Again, no real solutions. So n=1 doesn't work.

Next, n=2. The equation is:

16 + 15x + 15x² = 8

Rearranged:

15x² + 15x - 4 = 0

Using the quadratic formula:

x = [-15 ± sqrt(225 + 240)] / 30
= [-15 ± sqrt(465)] / 30

sqrt(465) is approximately 21.56, so x ≈ (-15 ± 21.56)/30. So two solutions: x ≈ (6.56)/30 ≈ 0.218 and x ≈ (-36.56)/30 ≈ -1.218. But since n=2, x must be in [2,3). The first solution 0.218 is less than 2, so discard. The second solution -1.218 is also less than 2. So no solutions here either.

Next, n=3. The equation is:

16

Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.


TrainOutput(global_step=100, training_loss=0.04211445438209921, metrics={'train_runtime': 10001.8669, 'train_samples_per_second': 0.04, 'train_steps_per_second': 0.01, 'total_flos': 0.0, 'train_loss': 0.04211445438209921, 'epoch': 0.007868439688409789})

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [32]:
text = "What is the sqrt of 101?"

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

INFO 10-16 02:59:20 [scheduler.py:2600] setting pause state to UNPAUSED


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 10-16 02:59:20 [jit_monitor.py:141] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 1/1 [00:10<00:00, 10.43s/it, est. speed input: 0.96 toks/s, output: 14.00 toks/s]


' - Answers\nMath and Arithmetic\nAlgebra\nWhat is the sqrt of 101?\nWiki User\n∙ 2011-06-06 01:26:51\nSee Answer\nBest Answer\nCopy\n10.049875621\nWiki User\n∙ 2011-06-06 01:26:51\nThis answer is:\nHelpful\nNot Helpful\n🙏\n0\n🤨\n0\n😮\n0\nAdd a Comment\nAdd your answer:\nEarn + 20 pts\nQ: What is the sqrt of 101?\nWrite your answer...\nSubmit'

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [33]:
model.save_lora("grpo_saved_lora")

Verify LoRA is actually trained!

In [34]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

Now we load the LoRA and test:

In [35]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

INFO 10-16 03:25:11 [scheduler.py:2600] setting pause state to UNPAUSED


Processed prompts: 100%|██████████| 1/1 [00:55<00:00, 55.14s/it, est. speed input: 1.05 toks/s, output: 20.46 toks/s]


"Okay, so I need to find the square root of 101. Hmm, let me think. I remember that square roots of non-perfect squares are irrational numbers, which means they can't be expressed as a simple fraction and their decimal representation goes on infinitely without repeating. So maybe I can estimate it?\n\nFirst, I should find two perfect squares between which 101 lies. The square roots of these perfect squares will be the integers between which the square root of 101 is. Let's see: 10^2 is 100, and 11^2 is 121. So 100 < 101 < 121. That means √100 < √101 < √121, so 10 < √101 < 11. So the square root of 101 is between 10 and 11.\n\nBut how close is it to 10 or 11? Maybe try plugging in decimals. Let me calculate 10.05^2. Wait, 10.05 squared would be 101.0025. Hmm, that's slightly higher than 101. So the square root is less than 10.05. Let me try 10.03. 10.03 squared is 100.6009, which is less than 101. So between 10.03 and 10.05. Let me try 10.04. So 10.04^2 = 100.8016, still below 101. Then 10.045. Calculating that, 10.045^2 is 100.902025, still under. Next, 10.047. Wait, maybe I'm overcomplicating this. Maybe I should see how many decimal places are needed for the exact value. But since this is a non-perfect square, the square root can't be expressed exactly as a decimal or fraction, so the answer would typically be left in the square root form or approximated to a certain degree of precision.\n\nSo unless there's a specific requirement for more decimal places, the square root of 101 is approximately 10.04987562. But in simplest form, it's just √101. Maybe I can factor 101 to see if it's a perfect square. Wait, 101 is a prime number. The factors of 101 are 1 and 101. So no, it's not a perfect square. Therefore, the square root remains irrational and can't be simplified further. So the answer is either √101 or approximately 10.04987562.\nTo find the square root of 101, we start by noting that it is a non-perfect square. This means that its square root is an irrational number, which cannot be expressed as a simple fraction or a terminating decimal.\n\nFirst, we identify the perfect squares closest to 101:\n\\[10^2 = 100\\]\n\\[11^2 = 121\\]\n\nSince \\(100 < 101 < 121\\), it follows that:\n\\[10 < \\sqrt{101} < 11\\]\n\nNext, we refine our estimate by checking values between 10 and 11. We start with 10.05:\n\\[10.05^2 = 101.0025\\]\nThis is slightly more than 101, so \\(\\sqrt{101}\\) is less than 10.05.\n\nWe then check 10.04:\n\\[10.04^2 = 100.8016\\]\nThis is less than 101, so \\(\\sqrt{101}\\) is greater than 10.04.\n\nNext, we try 10.045:\n\\[10.045^2 = 100.902025\\]\nThis is still less than 101, so \\(\\sqrt{101}\\) is greater than 10.045.\n\nFinally, we try 10.047:\n\\[10.047^2 = 100.943209\\]\nThis is still less than 101, so \\(\\sqrt{101}\\) is greater than 10.047.\n\nWe continue this process, but for practical purposes, it's often sufficient to use a calculator to find more precise values. Using a calculator, we find:\n\\[\\sqrt{101} \\approx 10.04987562\\]\n\nThus, the square root of 101 is:\n\\[\n\\boxed{\\sqrt{101}}\n\\]<end_working_out><SOLUTION>10.04987562</SOLUTION>"

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [36]:
# Merge to 16bit
if False: model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False:
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [37]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).